# Ensemble Classification Pipeline Example

This notebook demonstrates how to use the trained domain classification pipeline to make predictions.  
It supports both label prediction and probability estimation, with optional SHAP explanations.  
The first section shows how to classify a small batch of domains interactively;  
the second one computes performance metrics across the entire test dataset.


In [1]:
# Import necessary modules
import sys

from core.validator import load_saved_split, load_train_split, load_random_sample
from pipeline import DomainClassifier


Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
2026-04-01 12:09:18.123740: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-01 12:09:18.123770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-01 12:09:18.158182: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-01 12:09:18.240178: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other 

### Set label and dataset

In [ ]:
MALICIOUS_LABEL = "phishing"  # phishing / malware
STAGE = 1                     # 1 / 2 / 3
VERIFICATION = False           # True / False, use verification dataset of validation dataset

### Forced sequential cascade across aligned stage splits

This section evaluates the same test samples through Stage 1, then forwards only malicious false negatives to Stage 2, and finally to Stage 3. It assumes the saved stage splits are row-aligned, which should hold if all stage subsets were created from the same original dataframe and only feature columns were dropped before calling `train_test_split(...)`.


In [ ]:
import numpy as np
import pandas as pd
import contextlib
import io
from IPython.display import display
from sklearn.metrics import confusion_matrix, classification_report

# ------------------------------------------------------------
# Load all three aligned splits
# ------------------------------------------------------------
x1_test, y1_test = load_saved_split(1, MALICIOUS_LABEL, folder="./data/", verification=VERIFICATION)
x2_test, y2_test = load_saved_split(2, MALICIOUS_LABEL, folder="./data/", verification=VERIFICATION)
x3_test, y3_test = load_saved_split(3, MALICIOUS_LABEL, folder="./data/", verification=VERIFICATION)

print("Stage 1:", x1_test.shape, y1_test.shape)
print("Stage 2:", x2_test.shape, y2_test.shape)
print("Stage 3:", x3_test.shape, y3_test.shape)

assert len(y1_test) == len(y2_test) == len(y3_test), "Stage test splits have different lengths."
assert np.array_equal(y1_test, y2_test) and np.array_equal(y2_test, y3_test), (
    "Saved stage test splits do not appear to be aligned."
)
print("Stage splits appear aligned.")

# ------------------------------------------------------------
# Instantiate one classifier per stage
# ------------------------------------------------------------
clf_stage1 = DomainClassifier(data_sample=x1_test, label=MALICIOUS_LABEL)
clf_stage2 = DomainClassifier(data_sample=x2_test, label=MALICIOUS_LABEL)
clf_stage3 = DomainClassifier(data_sample=x3_test, label=MALICIOUS_LABEL)

THRESHOLD = 0.5

def to_binary_vector(y):
    y = np.asarray(y)
    if y.dtype.kind in {"i", "u", "b"}:
        return y.astype(int)
    if y.dtype.kind == "f":
        return np.rint(y).astype(int)
    normalized = np.array([str(v).strip().lower() for v in y], dtype=object)
    positive = {"1", "true", "malicious", "phishing", "malware"}
    return np.array([1 if v in positive else 0 for v in normalized], dtype=int)

@contextlib.contextmanager
def silence_stdout():
    with contextlib.redirect_stdout(io.StringIO()):
        yield

def classify_proba_batch(clf, X):
    X_stage = X
    partial_preds = []
    partial_pred_map = {}
    for arch, model in clf.base_models.items():
        preds = clf._predict_arch(model, X_stage, arch)
        preds = np.asarray(preds).reshape(-1)
        partial_preds.append(preds)
        partial_pred_map[arch] = preds
    meta_input = np.column_stack(partial_preds)
    meta_input_final = np.hstack((meta_input, X_stage[:, :10]))
    with silence_stdout():
        meta_proba = np.asarray(clf.meta_model.predict_proba(meta_input_final)).reshape(-1)
    fpd_proba = np.zeros_like(meta_proba, dtype=float)
    positive_mask = meta_proba > 1.0
    if np.any(positive_mask):
        with silence_stdout():
            fpd_scores = np.asarray(clf.fpd_model.predict_fp_proba(X_stage[positive_mask])).reshape(-1)
        fpd_proba[positive_mask] = fpd_scores
    return {
        "stage": clf.stage,
        "partial_preds": partial_pred_map,
        "meta_proba": np.round(meta_proba, 4),
        "fpd_proba": np.round(fpd_proba, 4),
        "final_proba": np.round(meta_proba.copy(), 4),
    }

# ------------------------------------------------------------
# REVERSED CASCADE (benigni pokracuji dal, maligni se zastavi)
#   Stage 1: vsechny vzorky
#     -> maligni (pred==1): zastavit, final_pred = 1
#     -> benigni (pred==0): postoupit do Stage 2
#   Stage 2: pouze Stage-1 benigni
#     -> maligni: zastavit, final_pred = 1
#     -> benigni: postoupit do Stage 3
#   Stage 3: pouze Stage-2 benigni
#     -> finalni rozhodnuti pro vsechny zbyvajici vzorky
# ------------------------------------------------------------
y = to_binary_vector(y1_test)
n = len(y)
final_pred = np.full(n, -1, dtype=int)

# Stage 1
stage1_out = classify_proba_batch(clf_stage1, x1_test)
stage1_proba = stage1_out["final_proba"]
stage1_pred = (stage1_proba >= THRESHOLD).astype(int)

malicious_stage1_idx = np.where(stage1_pred == 1)[0]
final_pred[malicious_stage1_idx] = 1
pass_to_stage2_idx = np.where(stage1_pred == 0)[0]

# Stage 2
stage2_proba = np.full(n, np.nan)
malicious_stage2_idx = np.array([], dtype=int)
pass_to_stage3_idx = np.array([], dtype=int)

if len(pass_to_stage2_idx) > 0:
    stage2_out = classify_proba_batch(clf_stage2, x2_test[pass_to_stage2_idx].copy())
    s2_proba = stage2_out["final_proba"]
    s2_pred = (s2_proba >= THRESHOLD).astype(int)
    stage2_proba[pass_to_stage2_idx] = s2_proba
    malicious_stage2_idx = pass_to_stage2_idx[s2_pred == 1]
    final_pred[malicious_stage2_idx] = 1
    pass_to_stage3_idx = pass_to_stage2_idx[s2_pred == 0]

# Stage 3
stage3_proba = np.full(n, np.nan)

if len(pass_to_stage3_idx) > 0:
    stage3_out = classify_proba_batch(clf_stage3, x3_test[pass_to_stage3_idx].copy())
    s3_proba = stage3_out["final_proba"]
    s3_pred = (s3_proba >= THRESHOLD).astype(int)
    stage3_proba[pass_to_stage3_idx] = s3_proba
    final_pred[pass_to_stage3_idx] = s3_pred

assert np.all(final_pred >= 0), "Nektery vzorek nema finalni predikci!"

# ------------------------------------------------------------
# Confusion matrix + metriky cele pipeline
# ------------------------------------------------------------
tn, fp, fn, tp = confusion_matrix(y, final_pred).ravel()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (tp + tn) / n

print("\n=== REVERSED CASCADE — vysledky cele pipeline ===")
print(f"Celkem vzorku   : {n}")
print(f"True maligni    : {int((y==1).sum())}")
print(f"True benigni    : {int((y==0).sum())}")
print()
print("--- Routing ---")
print(f"  Zastaveno na Stage 1 (maligni) : {len(malicious_stage1_idx)}")
print(f"  Postoupilo do Stage 2 (benigni): {len(pass_to_stage2_idx)}")
print(f"  Zastaveno na Stage 2 (maligni) : {len(malicious_stage2_idx)}")
print(f"  Postoupilo do Stage 3 (benigni): {len(pass_to_stage3_idx)}")
print()
print("--- Confusion matrix (pozitivni trida = maligni) ---")
print(f"  TP (spravne odhaleni maligni)  : {tp}")
print(f"  TN (spravne benigni)           : {tn}")
print(f"  FP (benigni spatne flagovani)  : {fp}")
print(f"  FN (maligni propusteny)        : {fn}")
print()
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 score  : {f1:.6f}")

print("\n=== sklearn classification_report ===")
print(classification_report(y, final_pred, target_names=["benign", MALICIOUS_LABEL]))

decided_at = np.zeros(n, dtype=int)
decided_at[malicious_stage1_idx] = 1
decided_at[malicious_stage2_idx] = 2
decided_at[pass_to_stage3_idx]   = 3

results_df = pd.DataFrame({
    "sample_idx"      : np.arange(n),
    "expected"        : y,
    "final_pred"      : final_pred,
    "decided_at_stage": decided_at,
    "stage1_proba"    : stage1_proba,
    "stage2_proba"    : stage2_proba,
    "stage3_proba"    : stage3_proba,
})
results_df["correct"] = (results_df["expected"] == results_df["final_pred"]).astype(int)

display(results_df.head(20))


Stage 1: (831, 62) (831,)
Stage 2: (831, 128) (831,)
Stage 3: (831, 176) (831,)
Stage splits appear aligned.
Stage: 1
📦 Loading model from models/cnn_stage_1_phishing_v1.1.keras


2026-04-01 12:09:28.623226: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-01 12:09:28.912719: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-01 12:09:28.915719: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

📦 Loading model from models/svm_stage_1_phishing_v1.1.pkl
📦 Loading model from models/XgBoost_stage_1_phishing_v1.1.xgb
📦 Loading model from models/Lgbm_stage_1_phishing_v1.1.pkl
📦 Loading model from models/feedforward_stage_1_phishing_v1.1.keras
Stage: 2
📦 Loading model from models/cnn_stage_2_phishing_v1.1.keras
📦 Loading model from models/svm_stage_2_phishing_v1.1.pkl
📦 Loading model from models/XgBoost_stage_2_phishing_v1.1.xgb
📦 Loading model from models/Lgbm_stage_2_phishing_v1.1.pkl
📦 Loading model from models/feedforward_stage_2_phishing_v1.1.keras
Stage: 3
📦 Loading model from models/cnn_stage_3_phishing_v1.1.keras
📦 Loading model from models/svm_stage_3_phishing_v1.1.pkl
📦 Loading model from models/XgBoost_stage_3_phishing_v1.1.xgb
📦 Loading model from models/Lgbm_stage_3_phishing_v1.1.pkl
📦 Loading model from models/feedforward_stage_3_phishing_v1.1.keras


2026-04-01 12:09:35.681916: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902


23/23 [==============================] - 0s 1ms/step

=== REVERSED CASCADE — vysledky cele pipeline ===
Celkem vzorku   : 831
True maligni    : 135
True benigni    : 696

--- Routing ---
  Zastaveno na Stage 1 (maligni) : 0
  Postoupilo do Stage 2 (benigni): 831
  Zastaveno na Stage 2 (maligni) : 107
  Postoupilo do Stage 3 (benigni): 724

--- Confusion matrix (pozitivni trida = maligni) ---
  TP (spravne odhaleni maligni)  : 106
  TN (spravne benigni)           : 693
  FP (benigni spatne flagovani)  : 3
  FN (maligni propusteny)        : 29

Accuracy  : 0.9615
Precision : 0.9725
Recall    : 0.7852
F1 score  : 0.8689

=== sklearn classification_report ===
              precision    recall  f1-score   support

      benign       0.96      1.00      0.98       696
    phishing       0.97      0.79      0.87       135

    accuracy                           0.96       831
   macro avg       0.97      0.89      0.92       831
weighted avg       0.96      0.96      0.96       831



,sample_idx,expected,final_pred,decided_at_stage,stage1_proba,stage2_proba,stage3_proba,correct
0,0,1,1,2,0.0,1.0000,NaN,1
1,1,0,0,3,0.0,0.0000,0.0000,1
2,2,0,0,3,0.0,0.0001,0.0001,1
3,3,1,1,2,0.0,1.0000,NaN,1
4,4,0,0,3,0.0,0.0001,0.0001,1
5,5,1,0,3,0.0,0.0111,0.0133,0
6,6,0,0,3,0.0,0.0000,0.0000,1
7,7,0,0,3,0.0,0.0000,0.0000,1
8,8,0,0,3,0.0,0.0001,0.0001,1
9,9,1,1,2,0.0,0.9999,NaN,1


### Save detailed cascade results


In [4]:

out_prefix = f"sequential_cascade_{MALICIOUS_LABEL}_{'verification' if VERIFICATION else 'validation'}"
results_path = f"./results/{out_prefix}_details.csv"
summary_path = f"./results/{out_prefix}_summary.csv"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"Saved detailed results to: {results_path}")
print(f"Saved summary to: {summary_path}")



NameError: name 'summary_df' is not defined